# Activity 2 — Scraping Economic Data


| § | Technique | Source |
|---|---|---|
| 1 | API with a key | FRED (US macro) |
| 2 | HTML tables → `pd.read_html` | Jamaica — quarterly GDP |
| 3 | Browser headers + SSL bypass | Jamaica — stock exchange index |
| 4 | XML / SDMX → `ElementTree` | Jamaica — CPI, then Suriname — government operations |
| 5 | Direct file download | Suriname — central bank `.xlsx` |
| 6 | Link harvesting → BeautifulSoup | EIA Brent, US Census trade |

In [ ]:
import io, os, re, json, glob, datetime, warnings
import requests
import pandas as pd
import numpy as np
from io import BytesIO, StringIO
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET

warnings.filterwarnings("ignore")           # keep the output readable
requests.packages.urllib3.disable_warnings()  # we deliberately skip some SSL checks below

# ---- Paths -----------------------------------------------------------------
# This notebook lives in workshop_code/activity/ (or solutions/), so the workshop
# folder is two levels up.
d          = os.getcwd()
PATH_RAW   = os.path.normpath(os.path.join(d, "..", "..", "raw"))
PATH_ACT2  = os.path.join(PATH_RAW, "act2")     # the bundled snapshots
os.makedirs(PATH_RAW, exist_ok=True)

# ---- The browser disguise --------------------------------------------------
# Many servers reject requests that do not look like a browser. Sending a
# User-Agent string is the single most useful line in this whole notebook.
hdr = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/58.0.3029.110 Safari/537")}

print("PATH_RAW  =", PATH_RAW)

# § 1 — FRED: API with a key

**Source shape:** the agency wants you to have the data and built a front door.

* [FRED](https://fred.stlouisfed.org/docs/api/fred/) — US macro and financial series
* [World Bank](https://datahelpdesk.worldbank.org/knowledgebase/topics/125589)
* [IMF](https://datahelp.imf.org/)

FRED needs a free key: register at
<https://fredredaccount.stlouisfed.org/apikeys> and paste it below. **Never commit a
key to a shared folder** — read it from an environment variable instead, as we do here.

In [ ]:
# Get a free key at https://fred.stlouisfed.org  ->  My Account  ->  API Keys.
# Best practice: set it once as an environment variable rather than typing it into
# a notebook that you might later share.
FRED_API_KEY = "72304a91a13cc0ea6c610ae5845c7493"
series = ["SP500", "DJIA", "NASDAQCOM"]

df = pd.DataFrame()
for serie in series :
    #&realtime_end=9999-12-31
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={serie}&api_key={FRED_API_KEY}&file_type=json"
    response = requests.get(url)
    response = response.json()
    dfi = pd.DataFrame(response['observations'])
    dfi = dfi[['value', 'date']]
    dfi.set_index('date', inplace=True)
    dfi.index = pd.to_datetime(dfi.index)
    dfi.rename(columns = { "value" : f"fred_{serie}" }, inplace=True)
    dfi[f"fred_{serie}"] = dfi[f"fred_{serie}"].apply(pd.to_numeric, errors='coerce')
    df = df.merge(dfi, left_index=True, right_index=True, how='outer')
    df = pd.DataFrame(df.resample("D").mean())
    #df = pd.concat([df, dfi], axis=1)
    df.sort_index(inplace=True)
    
df.to_csv(f"{PATH_RAW}/fred_data.csv")
print(f"FRED Updated: {df.index.max()}")
df.tail(5)

# § 2 — HTML tables: `pd.read_html`

**Source shape:** the numbers are in a `<table>` on a web page.

`pd.read_html(url)` fetches a page and returns a **list** of every table it found.
One line of download, then all the work is cleaning.

Our example is **Jamaica's quarterly GDP** from the Statistical Institute of Jamaica.
The raw table arrives in the shape statistical agencies love and analysts hate:

* industries down the rows, `2015 Q1`, `2015 Q2`, ... across the columns
* a banner row of metadata sitting above the real header
* names full of `&`, `,` and double spaces
* numbers stored as text, with `-` meaning "missing"

We want the opposite: a `DatetimeIndex` down the rows, one column per variable.
Getting between the two is `melt` → clean → `pivot`, and it is the most reusable
20 lines in this notebook.

In [ ]:
JM_GDP_URL = "https://www.datazoa.com/data/table.asp?a=view&th=69E287AF0E&dzuuid=1835&uid=dzadmin"
# The official page is https://statinja.gov.jm/NationalAccounting/Quarterly/NewQuarterlyGDP.aspx
# but it is an ASP form that does not respond to a plain GET; datazoa republishes the
# same table in a scrapeable form.

df = pd.read_html(JM_GDP_URL)[0]
df = df.rename(columns={"Unnamed: 0": "variable"})

print("Jamaica GDP:")
print(f"   shape: {df.shape}")
df.iloc[:4, :5]

### Step 1 — tidy the variable names

Good practice everywhere: strip punctuation, lowercase, collapse whitespace, join
words with underscores. `Total Value Added at Basic Prices` becomes
`total_value_added_at_basic_prices` — something you can type without quoting.

Each `.str.<method>()` applies to every value in the column, and they chain.

In [ ]:
# In Column variable: eliminate special symbols, extra spaces, use lowercase, and substitute spaces with underscores.
df["variable"] = df["variable"].str.replace(r"[^a-zA-Z0-9\s]", "", regex=True).str.lower().str.replace("  "," ").str.strip().str.replace(r"\s+", "_", regex=True)
df

### Step 2 — `melt`: wide to long

`pd.melt` "unpivots". Every date column becomes a row, so the table goes from
*18 rows × 41 columns* to *one row per (variable, date) pair*.

* `id_vars` — the column(s) to keep as identifiers
* `var_name` — name for the new column holding the old column **headers**
* `value_name` — name for the new column holding the **cell values**

In [ ]:
# Reshape dataframe from wide to long format.
df = pd.melt(df, id_vars=["variable"], var_name="date", value_name="value")
# Transform value in numeric, make nan those values that cannot be converted.
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df

### Step 3 — numbers, then dates

`errors="coerce"` is the important argument: instead of raising on `"-"` or `"n/a"`,
it writes `NaN` and moves on. Scraped tables are full of such sentinels.

Then the dates. They arrive as `2015 Q1`. Pandas cannot parse that, but the quarter
label maps cleanly onto the quarter's **first month** — `Q1 → 01`, `Q2 → 04`,
`Q3 → 07`, `Q4 → 10` — after which `"2015 01"` parses with `format="%Y %m"`.

In [ ]:
# drop missing values
df.dropna(subset=["value"], inplace=True)
df

In [ ]:
# Transform date into datetime format. Date has format 'Q1 2020' but we want it to be the first day of the first month in the quarter
df["date"] = df["date"].str.replace("Q1", "01").str.replace("Q2", "04").str.replace("Q3", "07").str.replace("Q4", "10")
df["date"] = pd.to_datetime(df["date"], format="%Y %m")
df

### Step 4 — `pivot`: long back to wide, the right way round

`pivot` is the inverse of `melt`. Now the *dates* become the index and the
*variables* become the columns — which is what every modelling tool expects.

In [ ]:
# Pivot data to have variables as columns.
df = df.pivot(index="date", columns="variable", values="value")

df.rename(columns={'total_value_added_at_basic_prices':'gdp'}, inplace=True)
df.insert(0, "gdp", df.pop("gdp"))
df

# § 3 — When the server does not want to talk to you

**Source shape:** a normal HTML table, behind a server that blocks robots.

Two arguments to `requests.get` solve most of these:

* `headers=hdr` — the browser disguise from the setup cell.
* `verify=False` — skip SSL certificate validation. Plenty of government sites run
  expired or self-signed certificates, and Python refuses them by default where a
  browser would just warn you.

**`verify=False` is a real security trade-off**, not a formality: you lose the
guarantee that you are talking to who you think you are. Use it for public
statistical tables. Never use it for anything authenticated.

Example: the **Jamaica Stock Exchange** main index.

Suppose we try the same method as before

In [ ]:
JSE_URL = "https://www.jamstockex.com/trading/indices/index-history/?indexCode=7&fromDate=2021-01-12"

df = pd.read_html(JSE_URL)[0]
df

This error states that you are forbidden to get the data.
Basically, your ping was incorrect.
The solution is to mask your request:

In [ ]:
JSE_URL = "https://www.jamstockex.com/trading/indices/index-history/?indexCode=7&fromDate=2021-01-12"

r = requests.get(JSE_URL, timeout=60, verify=False, headers=hdr)
df = pd.read_html(StringIO(r.text))[0]             # r.text = response decoded as string
df

Basic cleaning

In [ ]:

df.columns = df.columns.astype(str).str.lower()
df = df[["date", "value", "volume traded"]].set_index("date")
df = df.rename(columns={"value": "jse_index", "volume traded": "jse_volume"})
# format="mixed" copes with more than one date layout in the same column
df.index = pd.to_datetime(df.index, format="mixed")
df.sort_index().apply(pd.to_numeric, errors="coerce")


print("Jamaica Stock Exchange:")
df.tail(3)

# § 4 — XML and SDMX

**Source shape:** a machine-readable statistical exchange format.

Many central banks publish under the IMF's **e-GDDS** standard, which means SDMX-XML.
It is *designed* for machines to easily access and clean the data. The structure is always:

```xml
<Series INDICATOR="PCPI_IX" ...>
    <Obs TIME_PERIOD="2020-01" OBS_VALUE="103.4"/>
    <Obs TIME_PERIOD="2020-02" OBS_VALUE="104.1"/>
</Series>
```

so you walk the `<Series>` elements, filter on the indicator you want, and read the
attributes off each `<Obs>`. `.//Series` is XPath for "every `<Series>` anywhere in
the document".

First: **Jamaica's CPI**.

In [ ]:
JM_CPI_URL = "https://wups.statinja.gov.jm/wup/egddsfiles/CPI_Jamaica.xml"

r = requests.get(JM_CPI_URL, timeout=60, headers=hdr)
r.raise_for_status()                  # raise on any 4xx/5xx status code
root = ET.parse(BytesIO(r.content)).getroot()

rows = []
for s in root.findall(".//Series"):            # XPath: every <Series>, any depth
    if s.attrib.get("INDICATOR") == "PCPI_IX":  # consumer price index, all items
        for obs in s.findall("Obs"):            # direct <Obs> children
            v = obs.attrib.get("OBS_VALUE")
            if v is not None:
                rows.append((obs.attrib.get("TIME_PERIOD"), float(v)))

df = pd.DataFrame(rows, columns=["date", "cpi"])
df["date"] = pd.to_datetime(df["date"])
df.set_index("date").sort_index()

df

# § 5 — Excel Files: a file at a fixed URL

**Source shape:** the agency just puts a spreadsheet on the server and leaves it there.

The **Central Bank van Suriname** publishes its whole real-sector database as one
`.xlsx` at a stable address. No parsing, no headers, no disguise — download the bytes,
wrap them in `BytesIO` so pandas can treat them as a file, and open.

`pd.ExcelFile` opens the workbook *without* reading every sheet, so you can look at
`.sheet_names` first and pick. That matters here: the file has dozens of sheets and
they get renamed every year (`22.1 GDP (real) 2015-2025`), so **match the sheet name
with a pattern rather than hardcoding it**. Hardcoded sheet names are the single most
common way these scripts break in January.

In [ ]:
SR_XLSX_URL = "https://www.cbvs.sr/images/content/statistieken/Database/RealSectorStatistics.xlsx"

r = requests.get(SR_XLSX_URL, headers=hdr, timeout=120)
r.raise_for_status()
xls = pd.ExcelFile(BytesIO(r.content))         # bytes -> file-like -> workbook
xls

pandas reads the Excel but it is still not fully open: we need to locate the right sheet.

In [ ]:
# Match the sheet, do not hardcode it: "22.1 GDP (real) 2015-2025" and friends.
cand = [s for s in xls.sheet_names if "gdp" in s.lower() and "real" in s.lower()]
print(f"   candidate sheets: {cand}")
df = pd.read_excel(xls, sheet_name=sorted(cand)[-1], skiprows=3)
df

Now that the file is open, basic data cleaning follows:

In [ ]:
df = df.dropna(how="all", axis=0).dropna(how="all", axis=1)
df.columns = (df.columns.astype(str)
                .str.lower().str.strip()
                .str.replace(r"[^\w\s]", "", regex=True)
                .str.replace(r"\s+", "_", regex=True))
df = df.melt(id_vars="sector", var_name="year", value_name="value").dropna(subset=["value"])
df = df[df["sector"].astype(str).str.contains("Gross Domestic Product|GDP at market",
                                                case=False, na=False)]
df

In [ ]:
df["date"] = pd.to_datetime(df["year"].astype(str).str[:4] + "-12-01", errors="coerce")
df = df[['date', 'value']]
df

In [ ]:
df = df.dropna(subset=["date"]).set_index("date")[["value"]].rename(columns={"value": "gdp"})
df

# § 6 — Link harvesting with BeautifulSoup

**Source shape:** the data file exists, but its URL changes every release.

You cannot hardcode `.../data_2026Q1.xlsx` because next quarter it is `2026Q2`.
Instead scrape the *page*, collect every `<a href="...">`, and filter for the one
you want. The page layout is far more stable than the filenames.

```python
soup  = BeautifulSoup(r.content, "html.parser")
links = [a.get("href") for a in soup("a") if a.get("href")]
link  = [l for l in links if "xls" in l][0]
```

Examples: **EIA Brent crude** (oil price)

In [ ]:
url = "https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=RBRTE&f=D"
r = requests.get(url, verify=False, headers=hdr, timeout=60).content
soup = BeautifulSoup(r, "html.parser")

links = [a.get("href") for a in soup("a") if a.get("href")]
link  = [l for l in links if "xls" in l][0].replace("../", "")   # fix relative path
print(f"   harvested link: {link}")

xls = pd.ExcelFile(f"https://www.eia.gov/dnav/pet/{link}")
sheet = [s for s in xls.sheet_names if "data" in s.lower()][0]
df = pd.read_excel(xls, sheet_name=sheet, skiprows=2)

df.columns = df.columns.str.lower()
df = df.set_index("date")
# rename with a lambda: applies to every column name
df = df.rename(columns=lambda x: "brent price" if "brent" in x else x)
df.index = pd.to_datetime(df.index)
df